# Pandas - zadania 21-30

Zakres: nowe kolumny, tekst, braki danych i grupowanie (sekcje 5–7).

Rozwiązuj zadania w pustych komórkach z komentarzem `# tutaj wpisz rozwiązanie`. Gotowe komórki z danymi uruchom bez zmian.

### Import bibliotek i dane

Dane pobieramy z internetu (wymagane połączenie). Funkcja `wczytaj_csv_z_internetu` ma dwie wersje – wybierz wersję dla swojego systemu. Tabela `drinks` zawiera spożycie alkoholu w 193 krajach (porcje piwa, wódki i wina na osobę oraz litry czystego alkoholu).

In [1]:
import numpy as np
import pandas as pd
import requests
from io import StringIO

URL = 'https://raw.githubusercontent.com/justmarkham/pandas-videos/master/data/'

Uruchom **tylko jedną** z dwóch poniższych komórek – zależnie od systemu:

**Wersja dla Windows** – uruchom tę komórkę tylko na Windowsie (bezpośrednie `pd.read_csv(url)`):

In [ ]:
def wczytaj_csv_z_internetu(url, **kwargs):
    return pd.read_csv(url, **kwargs)

**Wersja dla macOS i Linux** – uruchom tę komórkę tylko na macOS lub Linuksie. Pobiera dane biblioteką `requests`, bo `pd.read_csv(url)` potrafi na macOS zgłosić błąd `CERTIFICATE_VERIFY_FAILED`. Ta wersja działa też na Windowsie, więc w razie kłopotów możesz jej użyć wszędzie:

In [4]:
def wczytaj_csv_z_internetu(url, **kwargs):
    odpowiedz = requests.get(url, timeout=30)
    odpowiedz.raise_for_status()
    return pd.read_csv(StringIO(odpowiedz.text), **kwargs)

In [5]:
drinks = wczytaj_csv_z_internetu(URL + 'drinks.csv')
drinks.head()

,country,beer_servings,spirit_servings,wine_servings,total_litres_of_pure_alcohol,continent
0,Afghanistan,0,0,0,0.0,Asia
1,Albania,89,132,54,4.9,Europe
2,Algeria,25,0,14,0.7,Africa
3,Andorra,245,138,312,12.4,Europe
4,Angola,217,57,45,5.9,Africa


### Ćwiczenie 21
Zrób kopię tabeli `drinks` (zmienna `napoje`). Dodaj kolumny:
- `porcje_razem` – suma porcji piwa, wódki i wina,
- `udział_wina` – udział wina w całkowitej liczbie porcji (zaokrąglony do 2 miejsc).

Wyświetl trzy kraje o największym udziale wina (kolumny `country` i `udział_wina`).

__Oczekiwany wynik:__

```
               country  udział_wina
55   Equatorial Guinea         0.72
171        Timor-Leste         0.67
92                Laos         0.66
```

__Wskazówka:__ Nowe kolumny tworzysz przez przypisanie z arytmetyką na kolumnach. Do wyboru największych wartości użyj `nlargest()`.

In [6]:
df = drinks.copy()
df['porcje razem'] = df['beer_servings'] + df['spirit_servings'] + df['wine_servings']
df['udział_wina'] = (df['wine_servings'] / df['porcje razem']).round(2)
df.sort_values('udział_wina', ascending=False)[['country', 'udział_wina']][:3]

,country,udział_wina
55,Equatorial Guinea,0.72
171,Timor-Leste,0.67
92,Laos,0.66


### Ćwiczenie 22
Podziel kraje na trzy kategorie według `total_litres_of_pure_alcohol`: `'małe'` (do 3 litrów włącznie), `'średnie'` (powyżej 3 do 9 litrów włącznie) i `'duże'` (powyżej 9). Policz, ile krajów jest w każdej kategorii.

__Oczekiwany wynik:__

```
total_litres_of_pure_alcohol
małe       84
średnie    75
duże       34
Name: count, dtype: int64
```

__Wskazówka:__ Użyj `pd.cut()` z parametrami _bins_ (np. `[-1, 3, 9, 20]`) i _labels_, a następnie `value_counts()`.

In [7]:
df2 = drinks.copy()
pd.cut(df2['total_litres_of_pure_alcohol'], bins=[-1, 3, 9, 20], labels=['małe', 'średnie', 'duże']).value_counts()

total_litres_of_pure_alcohol
małe       84
średnie    75
duże       34
Name: count, dtype: int64

### Ćwiczenie 23
Zamień pełne nazwy kontynentów na skróty: Africa – `AF`, Asia – `AS`, Europe – `EU`, North America – `NA`, Oceania – `OC`, South America – `SA`. Wynik zapisz w serii `skróty` i policz kraje w każdej grupie.

__Oczekiwany wynik:__

```
continent
AF    53
EU    45
AS    44
NA    23
OC    16
SA    12
Name: count, dtype: int64
```

__Wskazówka:__ Użyj metody `map()` ze słownikiem.

In [10]:
skróty = drinks['continent'].map({'Africa': 'AF', 'Asia': 'AS', 'Europe': 'EU',
                                       'North America': 'NA', 'Oceania': 'OC', 'South America': 'SA'})
skróty.value_counts()

continent
AF    53
EU    45
AS    44
NA    23
OC    16
SA    12
Name: count, dtype: int64

In [11]:
df3 = drinks.copy()
df3['continent'] = df['continent'].map(lambda x: x[:2].upper())
df3.value_counts('continent')

continent
AF    53
EU    45
AS    44
NO    23
OC    16
SO    12
Name: count, dtype: int64

### Ćwiczenie 24
Dla każdego kraju ustal, który z trzech napojów (piwo, wódka, wino) dominuje, czyli ma największą liczbę porcji. Wynik zapisz w kolumnie `dominujący` kopii tabeli (`napoje`) i policz, ile krajów przypada na każdy napój.

__Oczekiwany wynik:__

```
dominujący
beer_servings      114
spirit_servings     59
wine_servings       20
Name: count, dtype: int64
```

__Wskazówka:__ Użyj `apply(..., axis=1)` na trzech kolumnach z porcjami; funkcja dla wiersza może zwracać `wiersz.idxmax()`.

In [13]:
df4 = drinks.copy()
df4.apply(lambda row: row[['beer_servings', 'spirit_servings', 'wine_servings']].idxmax(), axis=1).value_counts()

beer_servings      114
spirit_servings     59
wine_servings       20
Name: count, dtype: int64

In [14]:
napoje = drinks.copy()
napoje['dominujący'] = napoje[['beer_servings', 'spirit_servings', 'wine_servings']].apply(lambda wiersz: wiersz.idxmax(), axis=1)
napoje['dominujący'].value_counts()

dominujący
beer_servings      114
spirit_servings     59
wine_servings       20
Name: count, dtype: int64

### Ćwiczenie 25
Tabela `telefony` zawiera modele i ceny zapisane jako tekst. Dodaj kolumny:
- `marka` – pierwsze słowo z nazwy modelu,
- `pamięć_gb` – liczba GB wyciągnięta z nazwy modelu (liczba całkowita),
- `cena_zł` – cena jako liczba całkowita.

__Oczekiwany wynik:__

```
                        model      cena    marka  pamięć_gb  cena_zł
0    Samsung Galaxy S23 128GB  3 499 zł  Samsung        128     3499
1       Apple iPhone 14 256GB  4 199 zł    Apple        256     4199
2  Xiaomi Redmi Note 12 128GB  1 299 zł   Xiaomi        128     1299
3       Apple iPhone 13 128GB  3 299 zł    Apple        128     3299
```

__Wskazówka:__ Użyj akcesora `.str`: `split()` z `.str[0]`, `extract()` z wyrażeniem regularnym `r'(\d+)GB'` (wynik to tabela, weź kolumnę `[0]`), `replace()` i `astype(int)`.

In [15]:
telefony = pd.DataFrame({'model': ['Samsung Galaxy S23 128GB', 'Apple iPhone 14 256GB', 'Xiaomi Redmi Note 12 128GB', 'Apple iPhone 13 128GB'],
                         'cena': ['3 499 zł', '4 199 zł', '1 299 zł', '3 299 zł']})
telefony

,model,cena
0,Samsung Galaxy S23 128GB,3 499 zł
1,Apple iPhone 14 256GB,4 199 zł
2,Xiaomi Redmi Note 12 128GB,1 299 zł
3,Apple iPhone 13 128GB,3 299 zł


In [16]:
telefony['marka'] = telefony['model'].str.split().str[0]
telefony['pamięć_gb'] = telefony['model'].str.split().str[-1].str.replace('GB', '')
telefony['cena_zł'] = telefony['cena'].str.replace('zł', '').str.replace(' ', '').astype(int)
telefony

,model,cena,marka,pamięć_gb,cena_zł
0,Samsung Galaxy S23 128GB,3 499 zł,Samsung,128,3499
1,Apple iPhone 14 256GB,4 199 zł,Apple,256,4199
2,Xiaomi Redmi Note 12 128GB,1 299 zł,Xiaomi,128,1299
3,Apple iPhone 13 128GB,3 299 zł,Apple,128,3299


### Ćwiczenie 26
Zrób kopię tabeli `drinks` (zmienna `napoje`) i dodaj kolumnę `piwosz` z wartością `'tak'` dla krajów z ponad 200 porcjami piwa i `'nie'` dla pozostałych. Zbuduj tabelę przestawną: wiersze to kontynenty, kolumny to wartości `piwosz`, a komórki to średnie spożycie wina (zaokrąglone do całości).

__Oczekiwany wynik:__

```
piwosz           nie    tak
continent                  
Africa          14.0   46.0
Asia             8.0   73.0
Europe         124.0  158.0
North America   19.0   43.0
Oceania         12.0  137.0
South America   73.0   31.0
```

__Wskazówka:__ Użyj `np.where()` oraz `pd.pivot_table(..., values=..., index=..., columns=..., aggfunc='mean')`.

In [17]:
df6 = drinks.copy()
df6['piwosz'] = df.apply(lambda x: 'tak' if x['beer_servings'] > 200 else 'nie', axis=1)
pd.pivot_table(df6, values='wine_servings', index='continent', columns='piwosz', aggfunc='mean').round(0)

piwosz,nie,tak
continent,,
Africa,14.0,46.0
Asia,8.0,73.0
Europe,124.0,158.0
North America,19.0,43.0
Oceania,12.0,137.0
South America,73.0,31.0


### Ćwiczenie 27
Tabela `dane` ma braki i zduplikowany wiersz.

a) Policz braki w każdej kolumnie.

b) Usuń zduplikowane wiersze, a następnie uzupełnij braki w kolumnie `wiek` medianą, a w kolumnie `pensja` średnią.

__Oczekiwany wynik:__

```
imię      0
wiek      1
pensja    3
dtype: int64
```

```
  imię  wiek       pensja
0  Ala  25.0  5200.000000
1  Ola  31.0  6433.333333
2  Ela  34.5  6100.000000
4  Jan  45.0  8000.000000
5  Ewa  38.0  6433.333333
```

__Wskazówka:__ Użyj `isna().sum()`, `drop_duplicates()` i `fillna()`. Wynik uzupełnienia przypisz z powrotem do kolumny.

In [18]:
dane = pd.DataFrame({'imię': ['Ala', 'Ola', 'Ela', 'Ola', 'Jan', 'Ewa'],
                     'wiek': [25, 31, np.nan, 31, 45, 38],
                     'pensja': [5200, np.nan, 6100, np.nan, 8000, np.nan]})
dane

,imię,wiek,pensja
0,Ala,25.0,5200.0
1,Ola,31.0,NaN
2,Ela,NaN,6100.0
3,Ola,31.0,NaN
4,Jan,45.0,8000.0
5,Ewa,38.0,NaN


In [20]:
dane.isna().sum()

imię      0
wiek      1
pensja    3
dtype: int64

In [21]:
dane = dane.drop_duplicates()
dane['wiek'] = dane['wiek'].fillna(dane['wiek'].median())
dane['pensja'] = dane['pensja'].fillna(dane['pensja'].mean())
dane

,imię,wiek,pensja
0,Ala,25.0,5200.000000
1,Ola,31.0,6433.333333
2,Ela,34.5,6100.000000
4,Jan,45.0,8000.000000
5,Ewa,38.0,6433.333333


### Ćwiczenie 28
Kolumna `odczyt` zawiera liczby zapisane jako tekst, ale też napisy `'brak'` i `'err'`.

a) Zamień kolumnę `odczyt` na liczby (nieczytelne wartości mają stać się brakami).

b) Policz braki.

c) Uzupełnij braki medianą kolumny.

__Oczekiwany wynik:__

```
  czujnik  odczyt
0       A    12.5
1       B     NaN
2       C    13.1
3       D     NaN
4       E    12.9
```

```
2
```

```
  czujnik  odczyt
0       A    12.5
1       B    12.9
2       C    13.1
3       D    12.9
4       E    12.9
```

__Wskazówka:__ Użyj `pd.to_numeric(..., errors='coerce')`, `isna().sum()` i `fillna()`.

In [22]:
pomiary = pd.DataFrame({'czujnik': ['A', 'B', 'C', 'D', 'E'], 'odczyt': ['12.5', 'brak', '13.1', 'err', '12.9']})
pomiary

,czujnik,odczyt
0,A,12.5
1,B,brak
2,C,13.1
3,D,err
4,E,12.9


In [23]:
pomiary['odczyt'] = pd.to_numeric(pomiary['odczyt'], errors='coerce')
pomiary

,czujnik,odczyt
0,A,12.5
1,B,NaN
2,C,13.1
3,D,NaN
4,E,12.9


In [24]:
print(pomiary['odczyt'].isna().sum())

2


In [25]:
pomiary['odczyt'] = pomiary['odczyt'].fillna(pomiary['odczyt'].median())
pomiary

,czujnik,odczyt
0,A,12.5
1,B,12.9
2,C,13.1
3,D,12.9
4,E,12.9


**Część d.** W serii `temperatura` brakuje pomiarów. Uzupełnij je **interpolacją liniową** (wartości leżące równomiernie między sąsiednimi pomiarami).

__Oczekiwany wynik:__

```
0    20.0
1    21.0
2    22.0
3    23.0
4    24.0
5    25.0
dtype: float64
```

__Wskazówka:__ Użyj metody `interpolate()`.

In [26]:
temperatura = pd.Series([20.0, np.nan, np.nan, 23.0, np.nan, 25.0])
temperatura

0    20.0
1     NaN
2     NaN
3    23.0
4     NaN
5    25.0
dtype: float64

In [27]:
temperatura.interpolate()

0    20.0
1    21.0
2    22.0
3    23.0
4    24.0
5    25.0
dtype: float64

### Ćwiczenie 29
Zbuduj tabelę podsumowującą kontynenty z kolumnami:
- `liczba_krajów` – liczba krajów,
- `średnie_piwo` – średnia liczba porcji piwa,
- `max_wino` – największa liczba porcji wina.

Posortuj ją malejąco według `średnie_piwo` i zaokrąglij wartości do 1 miejsca po przecinku.

__Oczekiwany wynik:__

```
               liczba_krajów  średnie_piwo  max_wino
continent                                           
Europe                    45         193.8       370
South America             12         175.1       221
North America             23         145.4       100
Oceania                   16          89.7       212
Africa                    53          61.5       233
Asia                      44          37.0       123
```

__Wskazówka:__ Użyj `groupby('continent').agg(nazwa=('kolumna', 'funkcja'), ...)`.

In [28]:
drinks.groupby('continent').agg(liczba_krajów=('country', 'count'),
                                   średnie_piwo=('beer_servings', 'mean'),
                                   max_wino=('wine_servings', 'max')).sort_values('średnie_piwo', ascending=False).round(1)

,liczba_krajów,średnie_piwo,max_wino
continent,,,
Europe,45,193.8,370
South America,12,175.1,221
North America,23,145.4,100
Oceania,16,89.7,212
Africa,53,61.5,233
Asia,44,37.0,123


### Ćwiczenie 30
Zrób kopię tabeli `drinks` (zmienna `napoje`). Dodaj kolumnę `średnia_kontynentu` ze średnią liczbą porcji piwa na kontynencie, do którego należy kraj, oraz kolumnę `różnica` (porcje piwa kraju minus średnia kontynentu, zaokrąglone do 1 miejsca). Wyświetl pięć krajów europejskich o największej różnicy (kolumny `country`, `beer_servings`, `średnia_kontynentu`, `różnica`).

__Oczekiwany wynik:__

```
            country  beer_servings  średnia_kontynentu  różnica
45   Czech Republic            361          193.777778    167.2
65          Germany            346          193.777778    152.2
98        Lithuania            343          193.777778    149.2
135          Poland            343          193.777778    149.2
81          Ireland            313          193.777778    119.2
```

__Wskazówka:__ Użyj `groupby('continent')['beer_servings'].transform('mean')`, a potem filtruj Europę i `nlargest()`.

In [30]:
napoje['średnia_kontynentu'] = napoje.groupby('continent')['beer_servings'].transform('mean')

In [31]:
napoje

,country,beer_servings,spirit_servings,wine_servings,total_litres_of_pure_alcohol,continent,średnia_kontynentu,różnica
0,Afghanistan,0,0,0,0.0,Asia,37.045455,-37.0
1,Albania,89,132,54,4.9,Europe,193.777778,-104.8
2,Algeria,25,0,14,0.7,Africa,61.471698,-36.5
3,Andorra,245,138,312,12.4,Europe,193.777778,51.2
4,Angola,217,57,45,5.9,Africa,61.471698,155.5
...,...,...,...,...,...,...,...,...
188,Venezuela,333,100,3,7.7,South America,175.083333,157.9
189,Vietnam,111,2,1,2.0,Asia,37.045455,74.0
190,Yemen,6,0,0,0.1,Asia,37.045455,-31.0
191,Zambia,32,19,4,2.5,Africa,61.471698,-29.5


In [29]:
napoje = drinks.copy()
napoje['średnia_kontynentu'] = napoje.groupby('continent')['beer_servings'].transform('mean')
napoje['różnica'] = (napoje['beer_servings'] - napoje['średnia_kontynentu']).round(1)
napoje[napoje['continent'] == 'Europe'].nlargest(5, 'różnica')[['country', 'beer_servings', 'średnia_kontynentu', 'różnica']]

,country,beer_servings,średnia_kontynentu,różnica
45,Czech Republic,361,193.777778,167.2
65,Germany,346,193.777778,152.2
98,Lithuania,343,193.777778,149.2
135,Poland,343,193.777778,149.2
81,Ireland,313,193.777778,119.2
